# Results tables

Turns the experiments' results into the paper's **LaTeX tables, printed inline**, so a
table can be re-read, adjusted and copied without leaving the notebook. Section 1 produces
Table 3 (average rank of the predictive methods), section 2 produces Table 4 (KS vs NLL win
counts with the cluster sign-flip test of Appendix H.4), and section 3 is a supplementary
ranking of SCORE within a kernel-score-only pool. Every
choice a table makes -- which data sets it pools, which arm realizes each row,
which metrics are columns -- is a literal in a configuration cell near the top;
nothing is inferred from a slug prefix or a run name.

Contents:

1. [Main table: mean rank across data sets](#1-main-table-mean-rank-across-data-sets)
2. [KS vs NLL: head-to-head win counts](#2-ks-vs-nll-head-to-head-win-counts)
3. [SCORE against the kernel-score baselines](#3-score-against-the-kernel-score-baselines)

Each section is self-contained and reads its own configuration, so adding one
never perturbs an existing one.

## 1. Main table: mean rank across data sets

**The question.** The per-data-set tables report scores, and scores are not
comparable across data sets -- the kernel score is $0.47$ on EUPPBench and
$0.0019$ on Depth, so an average over data sets is dominated by whichever one
has the largest units. A **rank** is unit-free, so it is the one summary that
can be pooled.

**What a cell is.** For task $t$ -- one data set in one mode at one horizon --
and metric $m$, every method present is ranked by its seed-mean score, $1$ =
best (all six metrics are lower-is-better). Two methods whose mean $\pm$ sd
intervals overlap, at the precision the per-task tables print them, count as
**equally good** and share the places they span: a
method's rank is one plus the methods that beat it plus half the methods it ties
with, so $0.500 \pm 0.020$ and $0.470 \pm 0.020$ tied for first both get
$1.5$. The ranks run over **every task**, but each **data
set counts once**: a method's ranks on the tasks of
data set $D$ are averaged first -- ETTm1's two modes at two horizons, the two
horizons of illness, weather, electricity and traffic; ERA5, Depth and EUPPBench
are one task each:

$$
r_{D,m}(a) \;=\; \frac{1}{|T_D(a,m)|} \sum_{t \in T_D(a,m)} \operatorname{rank}_{t,m}(a),
\qquad
R_m(a) \;=\; \frac{1}{|D(a,m)|} \sum_{D \in D(a,m)} r_{D,m}(a),
\qquad
\overline{R}(a) \;=\; \frac{1}{|P(a)|} \sum_{(D,m) \in P(a)} r_{D,m}(a) ,
$$

with $T_D(a,m)$ the tasks of $D$ where $a$ has a finished run scored on $m$,
$D(a,m)$ the data sets with at least one, and $P(a)$ every (data set, metric)
pair of $a$. $\overline{R}(a)$ is the final column: every per-data-set rank of
the method pooled over the metrics, so a metric scored on more data sets weighs
in proportion (every metric is scored on all eleven data sets; only Det and SB
lack the log score, so their $\overline{R}$ pools five metrics and every other
row's six). The printed table (1.7) shows all six columns and pools all six.

**Two properties of this summary to keep in view.** Both are made visible in
the output rather than argued away:

* A rank is **ordinal**. It says a method was beaten, never by how much, so a
  method losing every data set by $10^{-4}$ and one losing by a mile get the
  same row. The mean rank is a robustness statement, and the per-data-set
  scores in [1.9](#19-the-scores-behind-the-ranks) are what say whether a
  margin is real.
* A method absent from a data set is **ranked over fewer of them**, and the
  pool it is ranked against there is smaller too. That biases its mean
  downwards, so every cell carries the count it averaged over and the coverage
  matrix in [1.5](#15-coverage) prints the gaps in full.

### 1.1 Where the numbers come from

The source is W&B, read through `make_results_tables.py` -- the same module the
paper's per-data-set tables use, so a row here and a row there are the same
aggregation of the same runs. Two things make it the source rather than the
generated `tables/*.csv` in the results tree:

* **The CSVs cannot be joined on their own labels.** `spectral_diag` (the
  unnormalized transform-domain core) and `whitened_spectral_diag` (SCORE) both
  print `Fourier diagonal`, and on the ETT tables both carry the score label
  `KS`, so the pair `(core, score)` is genuinely duplicated there. W&B is keyed
  on the facet tuple `(method, transform, rank, stage1, stage2, loss)`, which
  is unique by construction.
* **The tags decide which runs a table shows.** ERA5 and Depth need
  `--require-tag main2026` -- without it earlier runs of the same
  configurations are pooled in and the seed counts come out above 5. The
  per-task `require_tags` below mirror `tools/generate_tables_*.sh` exactly.

EUPPBench and the time-series tasks (ETT, illness, weather, electricity, traffic) are the exceptions
and are read from `metrics.json` in the checkpoint tree, which carries
the same test scores. EUPPBench is not
facet-tagged in W&B at all; ETT is (`dataset:`/`features:`), but those two tags
alone cannot separate runs at different horizons, while the tree's
`<ds>/<mode>/T<horizon>/` paths can.

In [ ]:
import json
import sys
import pickle
import statistics
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown


def _project_root(start=None):
    """Nearest ancestor with a `.project-root` marker -- same rule plot_config uses."""
    start = Path.cwd() if start is None else Path(start)
    for parent in [start, *start.parents]:
        if (parent / ".project-root").exists():
            return parent
    return start


ROOT = _project_root()
NB_DIR = ROOT / "notebooks"
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

import make_results_tables as mrt   # facet parsing + seed aggregation, shared with the tables
import make_task_tables as mtt      # the tie rule of the per-task tables: equal within sd

#: The results tree.
from src.utils.paths import configured_path
SHARED = configured_path("results_dir")

#: Where a generated table is written by section 1.6.
TABLE_DIR = NB_DIR / "tables"

#: W&B is slow enough that re-running the notebook should not re-fetch. Delete
#: this file, or set REFRESH = True below, to pick up newly finished runs.
CACHE = NB_DIR / ".cache_results_tables.pkl"

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

### 1.2 Configuration

Everything the table depends on is in this cell and the next. To add a data set
that finishes later, append an entry to `TASKS`; nothing else has to change.

`d` is the target dimension, needed only to report the log score per
coordinate; it is taken from `mrt.DIMENSION` so it is defined in one place.

In [ ]:
#: ETT is the multivariate T96 task (seeds 1-5, the full grid), read from the
#: checkpoint tree: `fetch_dir(<ds>/multivariate, "T96")` walks
#: `<ds>/multivariate/T96/<slug>/`. The tree path carries the horizon, which the
#: W&B `dataset:` + `features:` tags alone do not. `TASKS` is one task per data
#: set: what the W&B cache holds and what section 3 ranks. The main table and the
#: win counts run over every task instead, `ALL_TASKS` below.
TASKS = [
    dict(key="etth1_M",   title="ETTh1",     dataset="T96",            features=None,
         source="tree",  require_tags=(), d=96 * 7,
         root=SHARED / "time_series" / "etth1" / "multivariate"),
    dict(key="etth2_M",   title="ETTh2",     dataset="T96",            features=None,
         source="tree",  require_tags=(), d=96 * 7,
         root=SHARED / "time_series" / "etth2" / "multivariate"),
    dict(key="ettm1_M",   title="ETTm1",     dataset="T96",            features=None,
         source="tree",  require_tags=(), d=96 * 7,
         root=SHARED / "time_series" / "ettm1" / "multivariate"),
    dict(key="ettm2_M",   title="ETTm2",     dataset="T96",            features=None,
         source="tree",  require_tags=(), d=96 * 7,
         root=SHARED / "time_series" / "ettm2" / "multivariate"),
    dict(key="era5",      title="ERA5",      dataset="era5",           features=None,
         source="wandb", require_tags=("main2026",)),
    dict(key="depth",     title="Depth",     dataset="depth_bts_crop", features=None,
         source="wandb", require_tags=("main2026",)),
    dict(key="euppbench", title="EUPPBench", dataset="euppbench",      features=None,
         source="tree",  require_tags=(), root=SHARED / "station_postprocessing"),
]
for _t in TASKS:
    if "d" not in _t:
        _t["d"] = mrt.DIMENSION[(_t["dataset"], _t["features"])]


#: **Every task**, which the main table (section 1) and the win counts (section 2)
#: both run over: the four ETT data sets in both modes at both horizons, both
#: horizons of illness, weather, electricity and traffic, and ERA5, Depth and
#: EUPPBench, one task each. Those three come from `runs` (the cache); every
#: time-series task is read from the tree. `cluster` is the data set a task
#: belongs to: the main table averages a method's ranks over a cluster's tasks
#: before it averages over clusters, and section 2 treats each cluster as one
#: unit of its p-values.
def _tree_task(key, title, cluster, d, root, horizon):
    return dict(key=key, title=title, cluster=cluster, d=d, root=root,
                dataset=f"T{horizon}")


ALL_TASKS = (
    [_tree_task(f"{ds}_{f}_T{T}", f"{name} {'uni' if f == 'S' else 'multi'} T{T}",
                name, T * (7 if f == "M" else 1), SHARED / "time_series" / ds / mode, T)
     for ds, name in (("etth1", "ETTh1"), ("etth2", "ETTh2"),
                      ("ettm1", "ETTm1"), ("ettm2", "ETTm2"))
     for f, mode in (("S", "univariate"), ("M", "multivariate"))
     for T in (96, 192)]
    + [_tree_task(f"illness_T{T}", f"Illness T{T}", "Illness", T * 7,
                  SHARED / "time_series" / "national_illness" / "multivariate", T)
       for T in (24, 36)]
    + [_tree_task(f"weather_T{T}", f"Weather T{T}", "Weather", T * 21,
                  SHARED / "time_series" / "weather" / "multivariate", T)
       for T in (96, 192)]
    + [_tree_task(f"electricity_T{T}", f"Electricity T{T}", "Electricity", T * 321,
                  SHARED / "time_series" / "electricity" / "multivariate", T)
       for T in (96, 192)]
    + [_tree_task(f"traffic_T{T}", f"Traffic T{T}", "Traffic", T * 862,
                  SHARED / "time_series" / "traffic" / "multivariate", T)
       for T in (96, 192)]
    + [dict(key=t["key"], title=t["title"], cluster=t["title"], d=t["d"],
            from_runs=t["key"])
       for t in TASKS if t["key"] in ("era5", "depth", "euppbench")]
)

#: Columns, in print order. All six are lower-is-better, which is what lets one
#: `ascending=True` rank serve every column.
METRICS = ["mse", "crps", "energy_score", "variogram_score",
           "kernel_score", "nll_per_d"]

METRIC_LABEL = {"mse": "MSE", "crps": "CRPS", "energy_score": "ES",
                "variogram_score": "VS", "kernel_score": "KS",
                "nll_per_d": r"NLL/$d$"}

#: The columns the LaTeX table prints, and the ones its Avg. pools: all six.
TABLE_METRICS = METRICS

#: Seeds a configuration needs before it is allowed to compete. Raise to 3 to
#: drop the thin cells instead of ranking them.
MIN_SEEDS = 1

#: Methods a (data set, metric) cell needs before its ranks count. A pool of one
#: is not a race: ranking a single method against nobody would hand it a free 1.0.
#: With the current runs this threshold drops nothing; it stays as a guard. Set
#: to 1 to keep pool-of-one cells.
MIN_POOL = 2

### 1.3 The rows

Each row names **one arm per data set**, by matching on facets rather than on a
label -- the label is what collides, the facets are what identify a run.

The fitting objective is fixed per row, because a score column only compares
like with like:

* the **covariance baselines**, Diag and LorD $r{=}1$, are in twice: their
  log-score-fitted arm (`NLL`) and, right below it, their kernel-score-fitted
  arm (`KS`), the layout of the per-task tables,
* **SCORE** is the two-stage fit with a stage-1 CRPS fit (`CRPS+KS`), which
  is the arm that exists on all eleven data sets; the transform differs by task
  (Fourier on the gridded and ETT tasks, the graph Fourier transform with a
  Matérn filter on the station network)
  and `method_prefix="whitened_"` is what lets one row pick up both,
* **Det** is fitted with MSE and **SB** with the
  kernel score, since neither has any other arm; they are point-forecast and
  sample-based reference rows, not competitors within an objective.

Mixing objectives down a column is the price of one table, and three of the
six columns charge it: `NLL/$d$`, `KS` and `MSE` are each some row's own
training objective. `NLL/$d$` is the sharpest case -- the SCORE row and
the two `KS` baselines are fitted with the kernel score, so their log score is
not what they optimized: on Depth SCORE comes out at $3.7\times10^{3}$, and the
`KS` baselines sit at $+43$ to $+300$ nats per coordinate on ETTm2
multivariate, weather and electricity, against at most $+4.3$ for the `NLL`
ones. `CRPS`, `ES` and `VS` are nobody's
objective here and are the columns that compare across the blocks without that
qualification. The printed table (1.7) shows all six, so its `MSE`, `KS` and
`NLL/$d$` columns, and the Avg. that pools them, carry the caveat.

`match` must select **exactly one** configuration per data set -- `score_frame`
raises if a row matches two, which is the check that a facet pattern has not
silently widened.

In [ ]:
def arm(method=None, method_prefix=None, rank="*", stage1="*", objective=None):
    """A predicate over `mrt.Config`. `"*"` means the facet is not constrained."""
    def match(c):
        if method is not None and c.method != method:
            return False
        if method_prefix is not None and not c.method.startswith(method_prefix):
            return False
        if rank != "*" and (c.rank or None) != rank:
            return False
        if stage1 != "*" and c.stage1 != stage1:
            return False
        if objective is not None and c.objective() != objective:
            return False
        return True
    return match


#: The table's blocks, in print order: the per-task tables' two.
BLOCK_PHYS = "Physical space"
BLOCK_SPEC = "Spectral space"
BLOCKS = [BLOCK_PHYS, BLOCK_SPEC]

METHODS = [
    dict(key="deterministic", label="Det", block=BLOCK_PHYS, fitted="MSE",
         match=arm(method="deterministic", objective="mse")),
    dict(key="phys_diag", label="Diag", block=BLOCK_PHYS, fitted="NLL",
         match=arm(method="phys_diag", rank=None, objective="nll")),
    dict(key="phys_diag_ks", label="Diag", block=BLOCK_PHYS, fitted="KS",
         match=arm(method="phys_diag", rank=None, objective="kernel_score")),
    dict(key="phys_lora_r1", label=r"LorD $r{=}1$", block=BLOCK_PHYS, fitted="NLL",
         match=arm(method="phys_lora", rank="1", objective="nll")),
    dict(key="phys_lora_r1_ks", label=r"LorD $r{=}1$", block=BLOCK_PHYS, fitted="KS",
         match=arm(method="phys_lora", rank="1", objective="kernel_score")),
    dict(key="generative", label="SB", block=BLOCK_PHYS, fitted="KS",
         match=arm(method="generative", objective="kernel_score")),
    dict(key="ours", label="SCORE", block=BLOCK_SPEC, fitted="CRPS+KS",
         match=arm(method_prefix="whitened_", rank=None, stage1="crps",
                   objective="kernel_score")),
]

#: Diag and LorD are in twice: the log-score fit and, right below it, the
#: kernel-score fit, the per-task tables' order, which also closes the
#: physical-space block with SB. `fitted` is the table's Objective column;
#: `ROW_NAME` keeps the two apart on screen.
ROW_NAME = {m["key"]: f'{m["label"]} ({m["fitted"]})' for m in METHODS}

#: Rows the ablations keep but the main table does not. Appending one back to
#: METHODS is all it takes to bring it in -- the ranks are recomputed against
#: whatever set is there, so a row is never ranked against a method the printed
#: table does not show.
#:
#:     dict(key="phys_lora_r16", label=r"LorD $r{=}16$", block=BLOCK_PHYS,
#:          fitted="NLL", match=arm(method="phys_lora", rank="16", objective="nll")),
#:     dict(key="phys_cholesky", label="Chol", block=BLOCK_PHYS,
#:          fitted="NLL", match=arm(method="phys_cholesky", rank=None, objective="nll")),
#:     dict(key="spectral_diag", label="Fourier diagonal", block=BLOCK_SPEC,
#:          fitted="NLL", match=arm(method="spectral_diag", rank=None, objective="nll")),

### 1.4 Reading the runs

`load_runs` groups every finished run of a task by its facet tuple, applying
that task's `require_tags`. The result is cached, since the W&B sweep takes a
few minutes; set `REFRESH = True` once after new runs finish.

In [ ]:
import contextlib
import io

REFRESH = False        # set True to re-query W&B instead of reading the cache


def load_runs(refresh=REFRESH):
    """`{task key: {facet tuple: [run entries]}}`, cached to disk."""
    if CACHE.is_file() and not refresh:
        return pickle.loads(CACHE.read_bytes())
    out = {}
    for t in TASKS:
        mrt.REQUIRE_TAGS.clear()
        mrt.REQUIRE_TAGS.update(t["require_tags"])
        try:
            if t["source"] == "tree":
                grouped = mrt.fetch_dir(str(t["root"]), t["dataset"])
            else:
                grouped = mrt.fetch(t["dataset"], t["features"])
        finally:
            mrt.REQUIRE_TAGS.clear()
        # Drop the wandb Run handles: they are not picklable and nothing below
        # needs them, the metrics were read out of the summary already.
        out[t["key"]] = {k: [{"cfg": e["cfg"], "seed": e["seed"],
                              "metrics": e["metrics"]} for e in v]
                         for k, v in grouped.items()}
    CACHE.write_bytes(pickle.dumps(out))
    return out


def load_all_runs(runs, tasks=ALL_TASKS):
    """`{task key: {facet tuple: [run entries]}}` for every task of `ALL_TASKS`.

    The tree tasks are read live -- local files, seconds, so there is nothing
    to cache -- and the one-task data sets reuse the (cached) `runs`.
    """
    out = {}
    for t in tasks:
        if "from_runs" in t:
            out[t["key"]] = runs[t["from_runs"]]
            continue
        mrt.REQUIRE_TAGS.clear()
        with contextlib.redirect_stdout(io.StringIO()):
            grouped = mrt.fetch_dir(str(t["root"]), t["dataset"])
        out[t["key"]] = {k: [{"cfg": e["cfg"], "seed": e["seed"],
                              "metrics": e["metrics"]} for e in v]
                         for k, v in grouped.items()}
    return out


runs = load_runs()
print({k: sum(len(v) for v in g.values()) for k, g in runs.items()}, "runs")
all_runs = load_all_runs(runs)
print(f"{len(all_runs)} tasks in "
      f"{len({t['cluster'] for t in ALL_TASKS})} data sets")

In [ ]:
#: The main table ranks every log score as measured. `mrt.aggregate` blanks a
#: configuration's whole log score once a seed passes `mrt.NLL_PER_D_LIMIT`,
#: which would take SCORE out of the Depth and ETTm2 rankings; the guard is
#: lifted here, and whether a cell competes is decided by its own spread instead
#: (`mtt.scored`: sd > |mean| is unstable, ties nothing and ranks last). `None`
#: restores `mrt`'s guard.
RANK_NLL_LIMIT = np.inf


def score_frame(runs, tasks=ALL_TASKS):
    """One row per (task, method, metric): seed mean, seed sd, seed count."""
    rows = []
    limit = mrt.NLL_PER_D_LIMIT
    if RANK_NLL_LIMIT is not None:
        mrt.NLL_PER_D_LIMIT = RANK_NLL_LIMIT
    try:
        for t in tasks:
            grouped = runs[t["key"]]
            for m in METHODS:
                hits = [(k, v) for k, v in grouped.items() if m["match"](v[0]["cfg"])]
                if len(hits) > 1:
                    raise ValueError(
                        f"row {m['key']!r} matches {len(hits)} configurations on "
                        f"{t['key']}: " + "; ".join(str(k) for k, _ in hits))
                if not hits:
                    continue
                facets, entries = hits[0]
                stats = mrt.aggregate(entries, t["d"])
                for metric in METRICS:
                    mean, sd, n = stats[metric]
                    rows.append(dict(task=t["key"], cluster=t["cluster"],
                                     method=m["key"], metric=metric, mean=mean,
                                     sd=sd, n_seeds=n, facets=facets))
    finally:
        mrt.NLL_PER_D_LIMIT = limit
    return pd.DataFrame(rows)


scores = score_frame(all_runs)
print(f"{len(scores)} (task, method, metric) cells, "
      f"{scores['mean'].notna().sum()} of them scored")

### 1.5 Coverage

The status of the table: which of the eleven data sets each row actually has an
arm on, and with how many seeds. A `0` is a gap that will make that row's mean
rank average over fewer data sets -- never a quietly interpolated number.

**No gap is left: every row has all eleven data sets (twenty-seven tasks) at
five seeds.** What remains is the log score of the two rows without a density
-- see below.

In [ ]:
def coverage(scores):
    """Seed count per (method, task), on the CRPS cell -- every arm is scored on it."""
    seeds = (scores[scores["metric"] == "crps"]
             .pivot_table(index="method", columns="task", values="n_seeds",
                          aggfunc="max"))
    order = [m["key"] for m in METHODS if m["key"] in seeds.index]
    cols = [t["key"] for t in ALL_TASKS if t["key"] in seeds.columns]
    out = seeds.loc[order, cols].fillna(0).astype(int)
    out.index = [ROW_NAME[i] for i in out.index]
    cluster = {t["key"]: t["cluster"] for t in ALL_TASKS}
    out["tasks"] = (out[cols] > 0).sum(axis=1)
    out["data sets"] = (out[cols] > 0).T.groupby(cluster).any().sum()
    return out


display(coverage(scores))

Which metrics are missing is worth seeing separately, because it decides which
columns a row's final average is taken over: `NLL/$d$` is undefined for the
Det and SB rows (neither has a density). That is the only gap left. The
variogram score on Depth, which the training-time evaluation skips because its
$O(d^2)$ pair sweep over $d = 238{,}560$ is too expensive, is computed post hoc
(unweighted, $p = 1$, over all 654 test images) and is in W&B and the results
tree for all Depth runs.

**Every log score is kept, and an unstable one ties nothing.** `mrt.aggregate` blanks a configuration's whole log
score once a seed passes $|\mathrm{nll}/d| > 10^{3}$; `score_frame` lifts that
guard (`RANK_NLL_LIMIT`), so every cell enters with its seed mean and every row
with a density is ranked on all eleven data sets. Whether a cell competes is
decided by its own spread rather than by a cutoff: a cell whose seed sd exceeds
its mean, as printed, is **unstable** -- its seeds do not agree even on its
size -- so it ties no stable cell and ranks behind all of them (`mtt.scored`).

That catches every cell the cutoff catches, SCORE on Depth
(one seed of five past $10^{3}$), ETTm2 multivariate (four of five at $T = 96$,
two at $T = 192$, $5 \times 10^{2}$ to $10^{8}$ nats per coordinate) and
electricity (all five at both horizons, $1.0 \times 10^{3}$ to
$1.5 \times 10^{5}$), and three it does not: SCORE on ETTh2
univariate $T = 192$ ($-0.099 \pm 0.387$), LorD $r{=}1$ (NLL) on Depth
($-0.058 \pm 1.080$) and LorD $r{=}1$ (KS) on ETTh1 univariate $T = 192$
($+0.022 \pm 0.037$). The last is a well-behaved score that happens to sit near
zero: a log score has no natural zero, so close to it the test is strict. The
kernel-score-fitted Diag and LorD reach $+300$ nats per coordinate on weather and
electricity, inflated but stable (sd well under the mean), so they are ranked as
measured. The same rule is used in the per-task tables, where otherwise
EUPPBench's Chol (KS) at $+331.6 \pm 413.0$ would tie the best log score and
take the bold.

In [ ]:
def metric_gaps(scores):
    """Data sets scored, per (method, metric)."""
    have = (scores[scores["mean"].notna() & (scores["n_seeds"] >= MIN_SEEDS)]
            .pivot_table(index="method", columns="metric", values="cluster",
                         aggfunc="nunique"))
    order = [m["key"] for m in METHODS if m["key"] in have.index]
    cols = [c for c in METRICS if c in have.columns]
    out = have.loc[order, cols].fillna(0).astype(int)
    out.index = [ROW_NAME[i] for i in out.index]
    return out


display(metric_gaps(scores))

### 1.6 Ranking

`rank_frame` ranks within each `(task, metric)` with the per-task tables' tie
rule (`mtt.shared_ranks`), so a task's ranks here are the ranks printed in its
table wherever the two pools coincide (the task tables add Chol on EUPPBench and
the univariate ETTh tasks). The overlap is read off the printed numbers, three
decimals after Depth's $\times 10^{3}$ (`PRINT_SCALE`), so every tie can be
checked against a table; and overlap is not transitive ($a \sim b$ and
$b \sim c$ without $a \sim c$), which counting pairs handles without picking
groups -- a pool's ranks still sum to $n(n+1)/2$. An unstable cell (1.5) ties
no stable one, however wide its sd -- the width is what would otherwise make it
tie everything: SCORE's Depth cell, $(3.7 \pm 8.0) \times 10^{3}$, would tie
every other method of its column.

`rank_frame` also records the size of the pool it ranked against, so a mean rank is never read without knowing how many
methods were in the race. Cells with fewer than `MIN_POOL` methods are dropped
and named as they go; with these seven rows **nothing is dropped**: with every
log score kept (1.5), every log-score cell is a pool of five.

`rank_norm` rescales each rank to $[0, 1]$ within its own pool; use it as
`column="rank_norm"` below when the pools differ enough that raw ranks are not
comparable across data sets -- here they run from five methods on the log
score to seven everywhere else, so it is worth looking at both.

In [ ]:
#: The scale the per-task tables print a metric at, which is where the tie rule
#: reads it: Depth's MSE, CRPS, VS and KS are x 10^3 there, and at 3 decimals
#: unscaled every one of them would print 0.002 and tie.
PRINT_SCALE = {"depth": mtt.DEPTH_SCALE}


def scored_cells(grp, task, metric):
    """`mtt.scored` cells of one pool's rows, at the scale `metric` prints on `task`."""
    scale = PRINT_SCALE.get(task, {}).get(metric, 1.0)
    return [mtt.scored(m, None if pd.isna(s) else s, scale)
            for m, s in zip(grp["mean"], grp["sd"])]


def rank_frame(scores, min_seeds=MIN_SEEDS, min_pool=MIN_POOL):
    """Rank within each (task, metric); 1 = best.

    Methods whose mean ± sd intervals overlap, as the task tables print them,
    are equally good and share the places they span (`mtt.shared_ranks`): two
    tied for first both get 1.5, as an exact tie would. A cell whose sd exceeds
    its mean is unstable: it ties no stable cell and ranks behind all of them.
    """
    ok = scores[scores["mean"].notna() & (scores["n_seeds"] >= min_seeds)].copy()
    ok["rank"] = np.nan
    for (task, metric), grp in ok.groupby(["task", "metric"]):
        ok.loc[grp.index, "rank"] = mtt.shared_ranks(scored_cells(grp, task, metric))
    ok["pool"] = ok.groupby(["task", "metric"])["mean"].transform("size")
    ok["rank_norm"] = (ok["rank"] - 1) / (ok["pool"] - 1).replace(0, np.nan)
    dropped = ok[ok["pool"] < min_pool]
    if len(dropped):
        for (t, m), grp in dropped.groupby(["task", "metric"]):
            print(f"dropped {t}/{m}: pool of {len(grp)} "
                  f"({', '.join(sorted(grp['method']))})")
    return ok[ok["pool"] >= min_pool].copy()


def rank_table(ranks, column="rank", metrics=METRICS):
    """Rows = methods, columns = `metrics`; the cell is `column` averaged over data sets.

    Each data set counts once: a method's ranks on the tasks of one data set
    (`cluster`) are averaged first, and a cell is the mean of those per-data-set
    ranks. `mean` pools every (data set, metric) rank of the method rather than
    averaging the columns, so a metric scored on more data sets weighs in
    proportion; it is taken over `metrics` only. `cov` counts data sets.
    """
    per_ds = (ranks.groupby(["cluster", "method", "metric"], as_index=False)
                   [column].mean())
    piv = per_ds.pivot_table(index="method", columns="metric", values=column,
                             aggfunc="mean")
    cov = per_ds.pivot_table(index="method", columns="metric", values=column,
                             aggfunc="size")
    order = [m["key"] for m in METHODS if m["key"] in piv.index]
    cols = [c for c in metrics if c in piv.columns]
    piv, cov = piv.loc[order, cols], cov.loc[order, cols].fillna(0).astype(int)
    per_ds = per_ds[per_ds["metric"].isin(cols)]
    piv["mean"] = per_ds.groupby("method")[column].mean().reindex(order)
    return piv, cov


ranks = rank_frame(scores)
piv, cov = rank_table(ranks)

_pretty = piv.copy()
_pretty.index = [ROW_NAME[i] for i in _pretty.index]
_pretty.columns = ["Avg." if c == "mean"
                   else METRIC_LABEL[c].replace("$", "").replace("\\", "")
                   for c in _pretty.columns]
display(_pretty.round(1))

The pool each rank was taken against, for reference: a row ranked 1st out of
three is not the same claim as 1st out of nine.

In [ ]:
display(ranks.pivot_table(index="task", columns="metric", values="pool", aggfunc="max")
             .reindex(index=[t["key"] for t in ALL_TASKS],
                      columns=[c for c in METRICS])
             .fillna(0).astype(int))

### 1.7 The LaTeX table

`to_latex_rank_table` prints the table (Table 3 of the paper) as `booktabs` LaTeX,
ready to paste: the rows split into a *Physical space* and a *Spectral space*
block as in the per-task tables, the paper's caption, and an `\adjustbox` wrapper
(the paper needs the `adjustbox` package, which the per-task tables already use).
The columns are `TABLE_METRICS`, all six, and Avg. pools those six -- five for Det
and SB, whose `NLL/$d$` cell prints `---`, so the table is the display of 1.6 in
print. The cells carry the mean rank and nothing else -- how many data sets each
was averaged over is in the coverage tables above, not in the printed table, so
read the two together.

**The bold is a coverage rule.** Only rows with a column's maximum coverage
compete for it, which stops a row ranked on fewer data sets from taking a column
from rows ranked on all of them. With every log score kept (1.5) every row has
full coverage in each column it has, so the rule is a guard rather than a filter.

Adjust the caption, the digits or the column set here and re-run this cell
alone; nothing above depends on it.

In [ ]:
#: The paper's caption wording, keep it verbatim.
CAPTION = (
    r"Ranked performance (averaged over all seeds and datasets) of the different"
    r" predictive methods. Best rank highlighted in bold."
)


def to_latex_rank_table(piv, cov, caption, label="main-rank", digits=1):
    """`booktabs` LaTeX: one block per `BLOCKS` entry, rows in `METHODS` order."""
    cols = [c for c in piv.columns if c != "mean"]
    width = len(cols) + 3
    # Only rows with a column's maximum coverage compete for the bold, so a row
    # ranked on fewer data sets cannot take a column from rows ranked on all of
    # them. With every log score kept (`score_frame`) every row has full
    # coverage in each column it has, so this is a guard, not a filter.
    full = {c: cov[c] == cov[c].max() for c in cols}
    full["mean"] = cov.max(axis=1) == cov.max(axis=1).max()
    best = {c: piv[c][full[c]].min() for c in list(cols) + ["mean"]}

    def cell(key, col):
        v = piv.loc[key, col]
        if pd.isna(v):
            return "---"
        body = f"{v:.{digits}f}"
        if abs(v - best[col]) < 10 ** -(digits + 3):
            body = rf"\mathbf{{{body}}}"
        return f"${body}$"

    lines = [
        r"\begin{table}[t]", r"\centering",
        rf"\caption{{{caption}}}", rf"\label{{tab:{label}}}", r"\small",
        r"\adjustbox{max width=\linewidth}{%",
        r"\begin{tabular}{ll" + "r" * (len(cols) + 1) + "}",
        r"\toprule",
        "Core & Objective & " + " & ".join(METRIC_LABEL[c] for c in cols)
        + r" & Avg. \\",
        r"\midrule",
    ]
    rows = [m for m in METHODS if m["key"] in piv.index]
    blocks = [b for b in BLOCKS if any(m["block"] == b for m in rows)]
    for bi, block in enumerate(blocks):
        if bi:
            lines.append(rf"\cmidrule(l){{1-{width}}}")
        lines.append(rf"\multicolumn{{{width}}}{{l}}{{\emph{{{block}}}}}\\")
        for m in rows:
            if m["block"] != block:
                continue
            cells = [cell(m["key"], c) for c in cols] + [cell(m["key"], "mean")]
            lines.append(rf"\quad {m['label']} & {m['fitted']} & "
                         + " & ".join(cells) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}", "}", r"\end{table}"]
    return "\n".join(lines)


#: The printed table ranks the same pools as 1.6 and shows `TABLE_METRICS`,
#: whose metrics its Avg. pools.
table_piv, table_cov = rank_table(ranks, metrics=TABLE_METRICS)
latex = to_latex_rank_table(table_piv, table_cov, CAPTION)
print(latex)

### 1.8 Writing it out

Written next to the runtime tables in `notebooks/tables/`. The results tree's
`tables/` directories are the output of `tools/generate_tables_*.sh` and are
per-experiment; this table spans experiments, so it does not belong in any one
of them and stays in the repository.

In [ ]:
TABLE_DIR.mkdir(parents=True, exist_ok=True)
path = TABLE_DIR / "main_rank.tex"
path.write_text(latex + "\n")
print(f"written to {path.relative_to(ROOT)}")

### 1.9 The scores behind the ranks

The ranks above are ordinal, so this is the table that says whether a margin is
real. One row per data set and metric, with the seed mean, the seed standard
deviation and the seed count; `rank` and `pool` are the columns section 1.6
built the table from.

Filter it to check any single cell, e.g.

```python
scores_wide.loc[("era5", "kernel_score")]
```

In [ ]:
scores_wide = (ranks
    .assign(label=lambda d: d["method"].map(ROW_NAME))
    .set_index(["task", "metric", "label"])
    [["mean", "sd", "n_seeds", "rank", "pool"]]
    .sort_index())
display(scores_wide.head(30))

## 2. KS vs NLL: head-to-head win counts

**The question.** Section 1 fixes one fitting objective per row, so even with
Diag and LorD $r{=}1$ in under both objectives the choice of objective shows up
there only as two ranks in a pool of seven, never measured head to head. This section
asks the complementary question directly: *given a covariance
parameterization, does fitting it with the kernel score or with the log
likelihood give the better forecast?* Every family below has both arms, so the
parameterization is held fixed and the objective is the only thing that
differs -- which is what makes the comparison a statement about the fit.

**What a comparison is.** For family $a$ (a covariance core at a fixed rank),
task $t$ (a data set at one horizon, in one mode) and metric $m$, the two arms'
seed-mean scores are compared and the lower one wins:

$$
w_m(a, t) \;=\; \mathbb{1}\!\left[\,\bar s^{\mathrm{KS}}_m(a,t)
                                  < \bar s^{\mathrm{NLL}}_m(a,t)\right],
\qquad
W_m(a) \;=\; \sum_{t \,\in\, T(a,m)} w_m(a,t),
$$

with $T(a,m)$ the tasks where **both** arms have a finished, scored run at
the required seed count. The printed cell is $W_m(a)$ over $|T(a,m)|$: the win
count and the number of comparisons it came out of, because $3$ wins says
nothing without knowing whether it was out of $3$ or out of $5$.

**Every task counts; the data set is the unit of the test.** The comparison runs
over all twenty-seven tasks
(`PAIR_TASKS`) rather than the rank tables' one task per data set: both horizons
of every time series, and both modes where a data set has them, so ETTm1
contributes four tasks and EUPPBench one. That makes the counts larger and makes
them *less* independent -- ETTm1's four tasks share one series, one backbone and
one recipe -- so a count is not tested as if its comparisons were coin flips.
[2.4](#24-the-win-counts) gives each count an exact **cluster sign-flip
p-value**, which swaps the two objectives' labels jointly for all comparisons of
a data set: eleven data sets give $2^{11} = 2048$ relabellings and a floor of
$p = 2/2048 \approx 0.001$, whatever the number of comparisons.

**Why these four metrics.** `NLL/$d$` and `KS` are left out on purpose -- they
are the two objectives being compared, so a win on either is the tautology
that a method scores best on what it was trained with. `MSE`, `CRPS`, `ES` and
`VS` are neutral: none of them is either arm's training loss, so a win on them
is about the fit rather than about the column.

**A win is ordinal**, the same caveat as section 1, and it bites harder here
because the pool is two: a win by $10^{-4}$ and a win by two orders of
magnitude both count $1$. [2.5](#25-the-margins-behind-the-counts) prints the
relative margin of every comparison and is the table to read before quoting a
count.

### 2.1 Configuration

A family is matched on the **grouping key** rather than on `cfg.rank`, because
`mrt.Config.key()` is where the rank-blind rule lives: a `phys_diag` run
may carry an inert `rank: 1` tag, and matching that tag would file it under no
family at all.

`phys_cholesky` is only defined where $\Sigma$ is small enough to store and
factor densely, which here is EUPPBench ($V = 115$) and the four univariate ETTh
tasks ($V = 96$, $192$), so it contributes five comparisons per metric from
three data sets -- too few for its own p-value to go below $2/2^3 = 0.25$.
Ranks not run under *both* objectives drop out on their own: a family with
one arm on a data set is not a comparison and is never counted as one. With
the current runs none is missing.

In [ ]:
from collections import defaultdict

#: One row of the win table per covariance family: the facet pattern that
#: names its two arms. `rank` is matched against the grouping key, which
#: `Config.key()` has already blanked for the rank-blind methods.
PAIR_FAMILIES = [
    dict(key="phys_diag",     label="Diag",               method="phys_diag",     rank=None),
    dict(key="phys_lora_r1",  label=r"LorD $r{=}1$",      method="phys_lora",     rank="1"),
    dict(key="phys_lora_r2",  label=r"LorD $r{=}2$",      method="phys_lora",     rank="2"),
    dict(key="phys_lora_r4",  label=r"LorD $r{=}4$",      method="phys_lora",     rank="4"),
    dict(key="phys_lora_r8",  label=r"LorD $r{=}8$",      method="phys_lora",     rank="8"),
    dict(key="phys_lora_r16", label=r"LorD $r{=}16$",     method="phys_lora",     rank="16"),
    dict(key="phys_cholesky", label="Chol",               method="phys_cholesky", rank=None),
]

#: The two arms, keyed by `Config.objective()` so the staged kernel-score
#: variants land in the KS arm exactly as they do in section 1.
FITS = {"kernel_score": "KS", "nll": "NLL"}

#: Metrics the wins are counted on, in print order. Neither objective is among
#: them; see the section header for why that is the point rather than a gap.
PAIR_METRICS = ["mse", "crps", "energy_score", "variogram_score"]

#: Compare the two arms on the seeds they *share* rather than on each arm's own
#: seed set -- see 2.2. Every task carries seeds 1-5 on every arm, so the
#: intersection is the full seed set and this changes nothing; it stays as a
#: guard.
SEED_MATCHED = True

#: Paired seeds a comparison needs before it counts. Every arm has five seeds,
#: so 3 drops nothing and only guards against incomplete arms.
PAIR_MIN_SEEDS = 3

#: The tasks the comparison runs over: **every task**, the same `ALL_TASKS` the
#: main table ranks. The unit that is counted is one (family, task, metric)
#: comparison, so ETTm1 enters with both horizons in both modes; but the tasks of
#: one data set are not independent evidence, so the p-values of 2.4 treat each
#: data set as one cluster (`cluster`).
PAIR_TASKS = ALL_TASKS
PAIR_CLUSTERS = list(dict.fromkeys(t["cluster"] for t in PAIR_TASKS))
CLUSTER_OF = {t["key"]: t["cluster"] for t in PAIR_TASKS}

#: Two-sided level at which a pooled win count is bolded in the LaTeX table.
PAIR_ALPHA = 0.05

### 2.2 One row per seed

The comparison is paired, so it is built from per-seed scores rather than from
the two arms' own aggregate means. Two things can make that necessary rather
than pedantic:

* **A repeated seed is not necessarily a replicate.** A W&B query filtered only
  on `dataset:` and `features:` can pool runs at different horizons under the
  same seed (e.g. a `prediction_length: 720` run in a table of $T = 96$ scores).
* **The two arms need not share a seed set.** If one arm carries an extra seed,
  an unmatched mean compares different seed sets.

Neither occurs with the time series read from the checkpoint tree
(`<ds>/<mode>/T<pred_len>/`, seeds 1-5 on every arm): the path carries the
horizon, and every family has the same five seeds under both objectives.
`SEED_MATCHED = True` is therefore a no-op on every task -- switching it off
flips none of the $668$ comparisons -- and is kept as a guard.

Runs sharing a seed within one configuration are collapsed to their mean here
and *reported* as they go, so such a case is never invisible.

In [ ]:
def arm_entries(grouped, family, objective):
    """The one configuration of `family` fitted with `objective`, or None."""
    hits = [(k, v) for k, v in grouped.items()
            if k[0] == family["method"] and k[2] == family["rank"]
            and v[0]["cfg"].objective() == objective]
    if len(hits) > 1:
        raise ValueError(
            f"family {family['key']!r} / {objective} matches {len(hits)} "
            "configurations: " + "; ".join(str(k) for k, _ in hits))
    return hits[0][1] if hits else None


def _seed_values(entries, metric, d):
    """`({seed: [value per run]}, seeds a guard removed)` for one metric.

    `nll_per_d` is the one metric not read off the summary as it stands: it is
    the log score over the target dimension, and it inherits `mrt`'s
    divergence guard, since seeds of one configuration differ only by
    initialization and a single diverging one says the density is unstable
    rather than that the seed was unlucky.
    """
    source = "nll" if metric == "nll_per_d" else metric
    per_seed = defaultdict(list)
    for e in entries:
        value = e["metrics"].get(source)
        if value is not None:
            per_seed[e["seed"]].append(value / d if metric == "nll_per_d"
                                       else value)
    if metric != "nll_per_d":
        return dict(per_seed), []
    bad = sorted(s for s, values in per_seed.items()
                 if any(abs(v) > mrt.NLL_PER_D_LIMIT for v in values))
    if not bad:
        return dict(per_seed), []
    if mrt.NLL_DROP_WHOLE_CONFIG:
        return {}, bad
    return {s: v for s, v in per_seed.items() if s not in bad}, bad


def seed_frame(runs, families=PAIR_FAMILIES, metrics=PAIR_METRICS,
               tasks=PAIR_TASKS):
    """One row per (task, family, fit, seed, metric): that seed's test score."""
    rows, repeats, dropped, seen = [], [], [], set()
    for t in tasks:
        grouped = runs[t["key"]]
        for fam in families:
            for objective, fit in FITS.items():
                entries = arm_entries(grouped, fam, objective)
                if entries is None:
                    continue
                for metric in metrics:
                    per_seed, bad = _seed_values(entries, metric, t["d"])
                    if bad:
                        dropped.append((t["key"], fam["key"], fit, bad))
                    for seed, values in sorted(per_seed.items()):
                        rows.append(dict(task=t["key"], family=fam["key"], fit=fit,
                                         seed=seed, metric=metric,
                                         value=float(np.mean(values)),
                                         n_runs=len(values)))
                        key = (t["key"], fam["key"], fit, seed)
                        if len(values) > 1 and key not in seen:
                            seen.add(key)
                            repeats.append((*key, metric, values))
    for task, fam, fit, seed, metric, values in repeats:
        spread = " vs ".join(f"{v:.4g}" for v in values)
        print(f"collapsed {len(values)} runs of seed {seed} in {task}/{fam}/{fit} "
              f"({metric} {spread})")
    for task, fam, fit, bad in dropped:
        print(f"dropped the log score of {task}/{fam}/{fit}: seed(s) "
              f"{', '.join(bad)} past |nll/d| > {mrt.NLL_PER_D_LIMIT:g}")
    return pd.DataFrame(rows)


pair_runs = all_runs
print(f"{len(pair_runs)} tasks in {len(PAIR_CLUSTERS)} data sets, "
      f"{sum(sum(len(v) for v in g.values()) for g in pair_runs.values())} runs")
seeds = seed_frame(pair_runs)
print(f"{len(seeds)} (task, family, fit, seed, metric) scores")

The seed sets themselves, so a mismatch is read off the table rather than
trusted: each cell is `KS seeds / NLL seeds`, and a cell where the two differ
is one where `SEED_MATCHED` is doing work. `--` is a family with no arm at all
on that task; a single number with no slash is a family with only one of
the two arms, which is exactly what "not counted in yet" looks like.

In [ ]:
def seed_sets(seeds, metric="crps"):
    """`KS/NLL` seed counts per (family, task) -- one metric, since all four
    are scored on the same runs."""
    n = (seeds[seeds["metric"] == metric]
         .groupby(["family", "task", "fit"])["seed"].nunique().unstack("fit"))
    out = pd.DataFrame(index=[f["key"] for f in PAIR_FAMILIES],
                       columns=[t["key"] for t in PAIR_TASKS], data="--")
    for (fam, task), row in n.iterrows():
        ks, nll = row.get("KS"), row.get("NLL")
        cell = ("--" if pd.isna(ks) and pd.isna(nll)
                else f"{int(ks)}/{int(nll)}" if not (pd.isna(ks) or pd.isna(nll))
                else f"KS {int(ks)}" if not pd.isna(ks) else f"NLL {int(nll)}")
        out.loc[fam, task] = cell
    out.index = [f["label"] for f in PAIR_FAMILIES]
    out.columns = [t["title"] for t in PAIR_TASKS]
    return out


display(seed_sets(seeds))

### 2.3 Pairing the arms

`pair_frame` turns the per-seed scores into one row per `(task, family,
metric)` comparison. A comparison exists only where both arms clear
`PAIR_MIN_SEEDS`; everything else is collected in `skipped` and printed, so a
missing comparison is always accounted for as either *one arm only* or *too few
seeds* and never as a silent zero. A metric that *neither* arm
was scored on leaves no row to skip and shows as `--` in the tables below;
with the current runs there is none.

`rel` is the margin as a fraction of the NLL arm's score, positive when KS
wins. It carries no weight in the counts -- it is what 2.5 reads to say whether
a counted win is a margin or a coin flip.

In [ ]:
def pair_frame(seeds, seed_matched=SEED_MATCHED, min_seeds=PAIR_MIN_SEEDS):
    """One row per (task, family, metric) where both arms are present."""
    rows, skipped = [], []
    for (task, family, metric), grp in seeds.groupby(["task", "family", "metric"]):
        arms = {fit: sub.set_index("seed")["value"]
                for fit, sub in grp.groupby("fit")}
        if len(arms) < 2:
            skipped.append(dict(task=task, family=family, metric=metric,
                                reason=f"only the {next(iter(arms))} arm"))
            continue
        ks, nll = arms["KS"], arms["NLL"]
        if seed_matched:
            common = sorted(set(ks.index) & set(nll.index))
            ks, nll = ks.loc[common], nll.loc[common]
        if min(len(ks), len(nll)) < min_seeds:
            skipped.append(dict(task=task, family=family, metric=metric,
                                reason=f"{len(ks)} KS and {len(nll)} paired "
                                       f"NLL seeds < {min_seeds}"))
            continue
        a, b = float(ks.mean()), float(nll.mean())
        rows.append(dict(task=task, family=family, metric=metric, ks=a, nll=b,
                         n_seeds=min(len(ks), len(nll)), delta=b - a,
                         rel=(b - a) / abs(b) if b else np.nan,
                         winner="KS" if a < b else "NLL" if b < a else "tie"))
    return pd.DataFrame(rows), pd.DataFrame(skipped)


pairs, skipped = pair_frame(seeds)
if len(skipped):
    for reason, grp in skipped.groupby("reason"):
        cells_ = ", ".join(f"{r.task}/{r.family}/{r.metric}"
                           for r in grp.itertuples())
        print(f"not compared -- {reason}: {cells_}")
print(f"\n{len(pairs)} comparisons, "
      f"{(pairs['winner'] == 'KS').sum()} won by KS, "
      f"{(pairs['winner'] == 'NLL').sum()} by NLL, "
      f"{(pairs['winner'] == 'tie').sum()} tied")

### 2.4 The win counts

The table the section is for: rows are the families, columns the four neutral
metrics, and the cell is *KS wins / comparisons*. The `All` row pools the
families and the `All` column pools the metrics; both are counts, so they
weight a data set by how many (family, task) pairs it carries -- four tasks for
each ETT data set, two for illness, weather, electricity and traffic, one for
ERA5, Depth and EUPPBench, and a seventh family (`Full covariance`) on EUPPBench
and the univariate ETTh tasks. That is where the column totals come from:
$6 \times 27 + 5 = 167$ comparisons on each of the four metrics.

The second table asks where the wins come from, counting over all tasks of each
data set; it is the one that shows whether a column's total is a consistent
effect or one data set's. The third gives each count its **cluster sign-flip
p-value** (`cluster_signflip_p`, two-sided): the net count
$S = \#\text{KS wins} - \#\text{NLL wins}$ against the $2^C$ values
$\sum_c \varepsilon_c S_c$ it takes when the two labels are swapped jointly within
each of the $C$ data sets. That respects the counting unit -- comparisons within
a data set may be dependent in any way -- and it is exact, not asymptotic. Its
price is resolution: with eleven data sets no count reaches below $p = 0.001$, and
one resting on three (the full covariance) cannot reach below $0.25$ however many
of its comparisons it wins. Treated as independent coin flips instead, $97$ of
$167$ would already clear a two-sided 5 % binomial test; the cluster test asks
that the data sets agree.

In [ ]:
def win_counts(pairs, index="family", metrics=PAIR_METRICS):
    """`(wins, n)`: KS wins and comparisons per cell, with `All` margins."""
    order = ([f["key"] for f in PAIR_FAMILIES] if index == "family"
             else PAIR_CLUSTERS if index == "cluster"
             else [t["key"] for t in PAIR_TASKS])
    d = pairs.assign(ks_win=(pairs["winner"] == "KS").astype(int),
                     cluster=pairs["task"].map(CLUSTER_OF))
    piv = lambda how: (d.pivot_table(index=index, columns="metric",
                                     values="ks_win", aggfunc=how)
                        .reindex(index=order, columns=metrics)
                        .fillna(0).astype(int))
    wins, n = piv("sum"), piv("size")
    wins.loc["All"], n.loc["All"] = wins.sum(), n.sum()
    wins["All"], n["All"] = wins[metrics].sum(axis=1), n[metrics].sum(axis=1)
    return wins, n


def win_table(wins, n, index="family"):
    """`wins / n` as strings; `--` where there is nothing to compare."""
    label = ({f["key"]: f["label"] for f in PAIR_FAMILIES} if index == "family"
             else {c: c for c in PAIR_CLUSTERS} if index == "cluster"
             else {t["key"]: t["title"] for t in PAIR_TASKS})
    out = pd.DataFrame(index=wins.index, columns=wins.columns, dtype=object)
    for i in wins.index:
        for c in wins.columns:
            out.loc[i, c] = ("--" if n.loc[i, c] == 0
                             else f"{wins.loc[i, c]}/{n.loc[i, c]}")
    out.index = [label.get(i, i) for i in out.index]
    out.columns = [c if c == "All"
                   else METRIC_LABEL[c].replace("$", "").replace("\\", "")
                   for c in out.columns]
    return out


import itertools


def cluster_signflip_p(sub):
    """Exact two-sided p of a net KS win count, the data sets as clusters.

    The statistic is the net win count `S = #KS wins - #NLL wins` over the
    comparisons in `sub`, which is what the tables count. Under the null the
    objective makes no difference, so the KS and NLL labels are exchangeable --
    but only **jointly within a data set**: its tasks share the series, the
    backbone and the training recipe, so its comparisons can be dependent in
    any way. Swapping the labels of a whole data set flips the sign of its net
    count `S_c` and leaves the dependence inside it intact, so the reference
    distribution is `sum_c eps_c S_c` over all `2^C` sign patterns `eps` of the
    `C` data sets, enumerated exactly. The smallest attainable p is `2 / 2^C`
    (0.0039 at nine data sets, 0.0078 at eight), however many comparisons each
    data set contributes -- which is the price of not counting ETTm1's four
    tasks as four independent witnesses.
    """
    sign = np.where(sub["winner"] == "KS", 1, np.where(sub["winner"] == "NLL", -1, 0))
    per = (pd.Series(sign, index=sub.index)
           .groupby(sub["task"].map(CLUSTER_OF)).sum().to_numpy(dtype=float))
    if not len(per):
        return np.nan
    null = np.array(list(itertools.product((1.0, -1.0), repeat=len(per)))) @ per
    return float(np.mean(np.abs(null) >= abs(per.sum()) - 1e-9))


def cluster_p_table(pairs, metrics=PAIR_METRICS):
    """Cluster sign-flip p per (family, metric), plus the pooled `All` row."""
    rows = [f["key"] for f in PAIR_FAMILIES] + ["All"]
    out = pd.DataFrame(index=rows, columns=list(metrics), dtype=float)
    for m in metrics:
        pm = pairs[pairs["metric"] == m]
        for r in rows:
            sub = pm if r == "All" else pm[pm["family"] == r]
            out.loc[r, m] = cluster_signflip_p(sub) if len(sub) else np.nan
    out.index = [{f["key"]: f["label"] for f in PAIR_FAMILIES}.get(i, i)
                 for i in out.index]
    out.columns = [METRIC_LABEL[c].replace("$", "").replace("\\", "")
                   for c in out.columns]
    return out


wins_fam, n_fam = win_counts(pairs, index="family")
wins_ds, n_ds = win_counts(pairs, index="cluster")

display(Markdown("**KS wins / comparisons, by covariance family**"))
display(win_table(wins_fam, n_fam, index="family"))
display(Markdown("**KS wins / comparisons, by data set** (all of its tasks)"))
display(win_table(wins_ds, n_ds, index="cluster"))
display(Markdown("**Cluster sign-flip p** (two-sided, data sets as clusters)"))
display(cluster_p_table(pairs).round(3))

Two patterns in those tables hold across the families, and both are read off the
by-data-set view rather than off the totals:

* **VS is the column the kernel score owns least** -- $139/167$ at $p = 0.005$,
  against $150/167$ on MSE, $140/167$ on CRPS and $156/167$ on ES, all three at
  the floor $p = 0.001$ because every data set nets a KS majority on them. On VS
  one does not: EUPPBench, where the NLL arm wins the variogram score on six of
  the seven families while losing MSE, CRPS and ES on all seven (ETTh1 is split,
  $15/26$). The split is the shape of the trade rather than noise. `MSE` sees
  only the mean and `CRPS` here is the per-coordinate score averaged over
  coordinates, so both read the *marginals*; the variogram score reads only the
  pairwise differences $|y_i - y_j|$, and two predictives with identical
  marginals are indistinguishable under CRPS but not under it
  (`src/evaluation/variogram.py`). The likelihood fit is buying its dependence
  structure with its marginal spread, and VS is the one column that does not
  charge it for that.
* **The gap widens with the rank** -- $63/108$ at `Diagonal`, $93/108$ at $r{=}1$,
  $97/108$ at $r{=}2$, $103/108$ at $r{=}4$ and $106/108$ at $r{=}8$ and $r{=}16$ -- and
  so does the evidence: the diagonal family is significant on no metric
  ($p = 0.17$–$0.42$), while every low-rank family is significant on every
  metric ($p \le 0.016$, the weakest being rank 1 on VS). One variance per
  coordinate leaves a
  proper score little to exploit and is the case where the likelihood is
  well conditioned; a high-rank likelihood fit is the case that comes apart. On
  ERA5 the low-rank NLL arms land at a CRPS of $0.63$ to $0.96$ against $0.15$
  for their KS twins, which is not a margin but a failure to fit. `Full
  covariance` wins $17/20$ but, resting on three data sets, cannot be
  significant ($p \ge 0.25$).

Of the $167$ (family, task) blocks, all of them scored on the four metrics, $126$
are KS sweeps and $5$ NLL sweeps -- the diagonal on ERA5, on ETTm1 multivariate
$T = 96$ and on both illness horizons, and rank 2 on ETTm1 multivariate
$T = 192$. Weather is the one data set KS sweeps outright ($48/48$), with ETTh2
($98/104$), ETTm2 ($92/96$), electricity ($46/48$; both NLL wins are the
diagonal's CRPS, by $4$–$6\%$) and traffic ($44/48$; all four NLL wins are the
diagonal's, CRPS by $16$–$18\%$ and VS by $3$–$5\%$, while every low-rank block
is a KS sweep) close behind. The likelihood holds on longest on
ETTm1 ($74/96$), illness ($34/48$) and ETTh1 ($86/104$ -- all of its NLL wins on
CRPS and VS, none on MSE or ES); everywhere its wins sit at the diagonal and the
low ranks, and they are gone from $r{=}8$ on except on the variogram score of
ETTh1 and EUPPBench. Depth brings six families and $24$ comparisons, $21$ of them
KS wins: the three NLL wins are all the diagonal's (MSE, ES and VS), and from
$r{=}1$ up the NLL arm's variogram score is $13$ to $900$ times the KS arm's.

### 2.5 The margins behind the counts

A count hides the size of what it counted, so these are the comparisons that
the counts rest on most lightly: the ten smallest $|rel|$, i.e. the ones a
rerun could plausibly flip. Read a win in 2.4 as an effect only if it is not on
this list.

`pairs` holds every comparison, so any single cell can be checked directly:

```python
pairs.query("task == 'euppbench' and metric == 'variogram_score'")
```

In [ ]:
def margin_view(frame):
    """The comparison frame with print labels and a percentage margin."""
    fam = {f["key"]: f["label"] for f in PAIR_FAMILIES}
    task = {t["key"]: t["title"] for t in PAIR_TASKS}
    out = frame.assign(
        task=frame["task"].map(task),
        family=frame["family"].map(fam),
        metric=frame["metric"].map(
            lambda m: METRIC_LABEL[m].replace("$", "").replace("\\", "")),
        **{"KS-NLL %": (100 * frame["rel"]).round(3)})
    return (out.set_index(["task", "family", "metric"])
               [["ks", "nll", "KS-NLL %", "n_seeds", "winner"]])


closest = pairs.reindex(pairs["rel"].abs().sort_values().index).head(10)
display(Markdown("**The ten closest comparisons** "
                 "(`KS-NLL %` = the NLL score KS saves, negative = NLL won)"))
display(margin_view(closest).round(3))

for side in ("KS", "NLL"):
    won = pairs[pairs["winner"] == side]
    if len(won):
        print(f"{side} wins: median margin "
              f"{100 * won['rel'].abs().median():.1f}% of the loser's score, "
              f"{len(won[won['rel'].abs() < 0.01])} of {len(won)} under 1%")

### 2.6 The LaTeX table

This is Table 4 of the paper.

The table the section exists to produce, **pooled over the
parameterizations**: one column per metric and a single row, whose cell is *KS
wins / comparisons* -- a comparison being one (parameterization, task) pair
where both arms are trained. The log-likelihood arm's count is the complement
of each cell whenever nothing ties, which is what the tie count in
[2.3](#23-pairing-the-arms) is there to confirm. A cell is **bold** when its
cluster sign-flip $p$ (2.4) is below `PAIR_ALPHA` $= 0.05$, and the $p$ values
are a second row, so the table states its test rather than leaving a count to
be read as one. All four cells are significant: MSE, CRPS and ES at the floor
$p = 0.001$, VS at $0.005$.

The metrics are deliberately **not** pooled into a total column: a count over
metrics weights MSE, CRPS, ES and VS equally, and they are neither independent
nor equally interesting -- VS alone scores the pairwise structure, and it is
the column that agrees least. The per-parameterization breakdown stays in
[2.4](#24-the-win-counts), where it is a diagnostic on this row rather than a
claim of its own.

In [ ]:
WINS_CAPTION = (
    r"Kernel score against log likelihood as the fitting objective, pooled over"
    r" the covariance parameterizations (Diag, LorD with $r = 1$--$16$, and"
    r" Chol) and over every task: both horizons of each time series,"
    r" univariate and multivariate where a data set has both, and ERA5, NYU Depth"
    r" v2 and EUPPBench. Each cell counts the (parameterization, task) pairs on"
    r" which the kernel-score arm beats the log-likelihood arm, out of the pairs"
    r" where both arms are trained; the parameterization is held fixed within a"
    r" comparison and the two arms are compared on the seeds they share. Neither"
    r" objective is among the metrics, so no column is either arm's own training"
    r" loss. The tasks of one data set are not independent, so $p$ is the exact"
    r" two-sided sign-flip test with the data sets as clusters: the two labels"
    r" are swapped jointly for all comparisons of a data set, over all $2^{C}$"
    rf" patterns of the $C = {len(PAIR_CLUSTERS)}$ data sets."
    r" \textbf{Bold}: $p < 0.05$."
)


def pooled_wins(pairs, metrics=PAIR_METRICS, sides=("KS",)):
    """`wins / comparisons` per metric, pooled over families and data sets."""
    metrics = list(metrics)
    n = pairs.groupby("metric").size().reindex(metrics).fillna(0).astype(int)
    out = pd.DataFrame(index=list(sides), columns=metrics, dtype=object)
    for side in sides:
        won = (pairs[pairs["winner"] == side].groupby("metric").size()
               .reindex(metrics).fillna(0).astype(int))
        for c in metrics:
            out.loc[side, c] = "--" if n[c] == 0 else f"{won[c]}/{n[c]}"
    out.index = [f"{s} wins" for s in out.index]
    out.columns = [METRIC_LABEL[c].replace("$", "").replace("\\", "")
                   for c in metrics]
    return out


def to_latex_win_table(pairs, caption, label="ks-vs-nll", metrics=PAIR_METRICS):
    """The pooled KS win counts as a single `booktabs` row."""
    cols = list(metrics)
    n = pairs.groupby("metric").size().reindex(cols).fillna(0).astype(int)
    won = (pairs[pairs["winner"] == "KS"].groupby("metric").size()
           .reindex(cols).fillna(0).astype(int))
    ties = pairs["winner"].eq("tie").sum()
    if ties:
        # The complement of a cell is then no longer the NLL arm's count, so
        # say so here rather than leaving the caption overclaiming.
        print(f"note: {ties} comparisons tied and are in neither count")
    p = {c: cluster_signflip_p(pairs[pairs["metric"] == c]) for c in cols}
    cells = ["---" if n[c] == 0 else
             (rf"$\mathbf{{{won[c]}/{n[c]}}}$" if p[c] < PAIR_ALPHA
              else f"${won[c]}/{n[c]}$") for c in cols]
    p_cells = ["---" if n[c] == 0 else f"${p[c]:.3f}$" for c in cols]
    return "\n".join([
        r"\begin{table}[ht]", r"\centering",
        rf"\caption{{{caption}}}", rf"\label{{tab:{label}}}", r"\small",
        r"\setlength{\tabcolsep}{6pt}",
        r"\begin{tabular}{l" + "r" * len(cols) + "}",
        r"\toprule",
        "Fit & " + " & ".join(METRIC_LABEL[c] for c in cols) + r" \\",
        r"\midrule",
        "KS wins & " + " & ".join(cells) + r" \\",
        "$p$ & " + " & ".join(p_cells) + r" \\",
        r"\bottomrule", r"\end{tabular}", r"\end{table}",
    ])


display(pooled_wins(pairs))

wins_latex = to_latex_win_table(pairs, WINS_CAPTION)
print(wins_latex)

TABLE_DIR.mkdir(parents=True, exist_ok=True)
wins_path = TABLE_DIR / "ks_vs_nll_wins.tex"
wins_path.write_text(wins_latex + "\n")
print(f"\nwritten to {wins_path.relative_to(ROOT)}")

## 3. SCORE against the kernel-score baselines

**The question.** Section 1 carries both the log-score and the kernel-score fits
of Diag and LorD $r{=}1$, ranked against every other row. Section 2 measured what
the objective alone is worth ($585$ of $668$ comparisons over every task), which
makes the matched question the interesting one: **among arms that were all fitted
with the kernel score, where does SCORE rank?** This section asks it in a pool of
kernel-score arms only, one task per data set, with Chol where it exists.

**The pool.** SCORE -- stage-1 CRPS, stage-2 kernel score, Fourier on the gridded
and ETT tasks and the graph Fourier transform with a Matérn filter on the station
network -- against Diag (`phys_diag`), LorD at $r = 1$ (`phys_lora`) and Chol
(`phys_cholesky`), each in its kernel-score arm. That is three arms on the four
ETT tasks, ERA5 and Depth, and four on EUPPBench, the only data set where a full
covariance is tractable. The second table adds **SB**, which is fitted with the
same kernel score but is nonparametric: a reference of a different kind rather
than another covariance core, so it gets its own pool instead of being mixed into
the first. It has an arm on every data set, which makes that pool four everywhere
but EUPPBench, where it is five.

**What a cell is.** Seed-mean scores are ranked within each (data set, metric),
$1$ = best, with arms whose mean $\pm$ sd intervals overlap sharing their
places, exactly as in section 1. The single
row is that rank averaged over the data sets; the marginal table underneath is
the rank itself, as `rank/pool`. Seeds are matched across the pool of a cell,
for the reason [2.2](#22-one-row-per-seed) gives; with every arm at seeds 1-5
that is currently a no-op.

### 3.1 Configuration

`ks_arm` differs from section 1's `arm` in two ways: the objective is pinned to
the kernel score, since the whole point is a within-objective ranking, and the
rank is read off the grouping key rather than off `cfg.rank`, which is what
keeps an inert `rank: 1` tag on a rank-blind method from hiding it.

`sample` is what separates the two tables, so moving SB into
or out of the first pool is a one-word edit.

In [ ]:
def ks_arm(method=None, method_prefix=None, rank=None, stage1="*"):
    """A predicate over `(grouping key, Config)` selecting one kernel-score arm."""
    def match(key, cfg):
        if method is not None and cfg.method != method:
            return False
        if method_prefix is not None and not cfg.method.startswith(method_prefix):
            return False
        if key[2] != rank:
            return False
        if stage1 != "*" and cfg.stage1 != stage1:
            return False
        return cfg.objective() == "kernel_score"
    return match


#: The ranked pool. Every member is fitted with the kernel score, so a rank
#: here compares parameterizations rather than objectives. `sample` marks the
#: nonparametric arm, which the first table leaves out.
KS_POOL = [
    dict(key="ours", label="SCORE", sample=False,
         match=ks_arm(method_prefix="whitened_", stage1="crps")),
    dict(key="phys_diag", label="Diag", sample=False,
         match=ks_arm(method="phys_diag")),
    dict(key="phys_lora_r1", label=r"LorD $r{=}1$", sample=False,
         match=ks_arm(method="phys_lora", rank="1")),
    dict(key="phys_cholesky", label="Chol", sample=False,
         match=ks_arm(method="phys_cholesky")),
    dict(key="generative", label="SB", sample=True,
         match=ks_arm(method="generative")),
]

#: The row the tables report. Every other pool member is there to be ranked
#: against, not printed.
FOCUS = "ours"

### 3.2 The seed means and the ranks

`pool_scores` averages over seeds first, keeping the seed sd for the tie rule,
and `focus_ranks` ranks the averages,
in that order -- the reverse (ranking per seed, then averaging the ranks) is a
different statistic and not the one section 1 reports. Both tables are built
from the same frame, so the only thing that separates them is which pool the
ranking saw.

`ks_scores` keeps the seed means and sds and `ks_ranks_*` the per-cell rank, gap and
pool, so any cell in the tables below can be checked directly:

```python
ks_scores.query("task == 'euppbench' and metric == 'crps'")
ks_ranks_cores.sort_values("gap", ascending=False).head()
```

In [ ]:
def pool_entries(grouped, member):
    """The one configuration of `member` in a task's runs, or None."""
    hits = [(k, v) for k, v in grouped.items() if member["match"](k, v[0]["cfg"])]
    if len(hits) > 1:
        raise ValueError(
            f"pool member {member['key']!r} matches {len(hits)} configurations: "
            + "; ".join(str(k) for k, _ in hits))
    return hits[0][1] if hits else None


def pool_scores(runs, pool, metrics=PAIR_METRICS, seed_matched=SEED_MATCHED,
                min_seeds=PAIR_MIN_SEEDS):
    """One row per (task, metric, method): the seed mean and sd of that arm."""
    rows, thin = [], []
    for t in TASKS:
        grouped = runs[t["key"]]
        for metric in metrics:
            per_method = {}
            for member in pool:
                entries = pool_entries(grouped, member)
                if entries is None:
                    continue
                per_seed, _ = _seed_values(entries, metric, t["d"])
                if per_seed:
                    per_method[member["key"]] = {s: float(np.mean(v))
                                                 for s, v in per_seed.items()}
            if seed_matched and per_method:
                common = set.intersection(*(set(v) for v in per_method.values()))
                per_method = {k: {s: v for s, v in d.items() if s in common}
                              for k, d in per_method.items()}
            thin += [(t["key"], metric, k, len(d)) for k, d in sorted(per_method.items())
                     if len(d) < min_seeds]
            per_method = {k: d for k, d in per_method.items() if len(d) >= min_seeds}
            if len(per_method) < 2:
                continue
            for k, d in sorted(per_method.items()):
                values = list(d.values())
                rows.append(dict(task=t["key"], metric=metric, method=k,
                                 mean=float(np.mean(values)),
                                 sd=statistics.stdev(values) if len(values) > 1
                                 else None,
                                 n_seeds=len(d)))
    for task, metric, k, n in thin:
        print(f"dropped {task}/{metric}/{k}: {n} matched seeds < {min_seeds}")
    return pd.DataFrame(rows)


def focus_ranks(scores, focus=FOCUS):
    """Per (task, metric): `focus`'s rank, the pool it was ranked in, and its
    gap to the best arm as a fraction of that arm's score.

    The rank is section 1's: arms equal within their sd share their places.
    """
    rows = []
    for (task, metric), grp in scores.groupby(["task", "metric"]):
        mean = grp.set_index("method")["mean"]
        if focus not in mean.index:
            continue
        rank = dict(zip(grp["method"],
                        mtt.shared_ranks(scored_cells(grp, task, metric))))
        rows.append(dict(task=task, metric=metric,
                         rank=float(rank[focus]),
                         pool=len(mean), n_seeds=int(grp["n_seeds"].min()),
                         best=mean.idxmin(),
                         gap=float((mean[focus] - mean.min()) / abs(mean.min()))))
    return pd.DataFrame(rows)


CORES = [m for m in KS_POOL if not m["sample"]]

ks_scores = pool_scores(runs, KS_POOL)
ks_ranks_cores = focus_ranks(pool_scores(runs, CORES))
ks_ranks_sampler = focus_ranks(ks_scores)
print(f"{len(ks_ranks_cores)} ranked cells against the cores, "
      f"{len(ks_ranks_sampler)} with the sample-based arm")

### 3.3 The two tables

The single row is the rank averaged over the data sets; the marginal table is
the rank in each one, printed as `rank/pool` because the pool is not constant
-- `Full covariance` exists only on EUPPBench and the second table adds the
sample-based arm, so a $3$ out of $3$ and a $3$ out of $5$ are different
statements. `--` would be a metric the data set does not score; with the
current runs there is none.

In [ ]:
def rank_row(ranks, label="SCORE", metrics=PAIR_METRICS):
    """Single row: `FOCUS`'s rank averaged over the data sets, per metric."""
    metrics = list(metrics)
    mean = ranks.groupby("metric")["rank"].mean().reindex(metrics)
    n = ranks.groupby("metric")["rank"].size().reindex(metrics).fillna(0).astype(int)
    return pd.DataFrame([["--" if n[c] == 0 else f"{mean[c]:.1f}" for c in metrics]],
                        index=[label],
                        columns=[METRIC_LABEL[c].replace("$", "").replace("\\\\", "")
                                 for c in metrics])


def rank_marginal(ranks, metrics=PAIR_METRICS):
    """Rows = data sets, columns = metrics; the cell is `rank/pool`."""
    metrics = list(metrics)
    out = pd.DataFrame("--", index=[t["key"] for t in TASKS], columns=metrics,
                       dtype=object)
    for r in ranks.itertuples():
        out.loc[r.task, r.metric] = f"{r.rank:g}/{r.pool}"
    out.index = [t["title"] for t in TASKS]
    out.columns = [METRIC_LABEL[c].replace("$", "").replace("\\\\", "")
                   for c in metrics]
    return out


for title, ranks in (("Gaussian cores only", ks_ranks_cores),
                     ("with the sample-based arm", ks_ranks_sampler)):
    display(Markdown(f"**Mean rank of SCORE, {title}**"))
    display(rank_row(ranks))
    display(Markdown(f"**Per data set, {title}** (`rank/pool`)"))
    display(rank_marginal(ranks))

Those mean ranks are middling, and the reason they are is something a rank
cannot show: SCORE is not mid-pack anywhere. Read with the tie rule, it is first
on three data sets and behind on two, and on the remaining two the whole pool
sits inside its own seed spread.

* **ETTh1, ETTm1 and ERA5: first.** Outright on CRPS and VS on all three, and on
  ES on ETTh1 and ERA5; MSE on ETTh1 and ERA5 and ES on ETTm1 are shared firsts
  (tied with one core, ahead of the other, $1.5$), and ETTm1's MSE ties all three
  cores. On ERA5 not narrowly -- the CRPS is $0.1442$ against $0.1548$ for the
  best baseline and the variogram score $0.0671$ against $0.0790$, margins of
  $6.8\%$ and $15.0\%$. On ETTm1 the margins are $3.7\%$ (CRPS), $2.7\%$ (VS),
  $1.5\%$ (MSE) and $0.4\%$ (ES), the last two inside the seed spread.
* **Depth: last of three on MSE, CRPS and ES**, by $4.9\%$, $7.5\%$ and $5.7\%$,
  each clear of the sd. On VS the pool sits inside $1.9\%$ and the three tie at
  $2$.
* **EUPPBench: third of four on MSE, CRPS and ES** -- $1.7\%$, $0.7\%$ and
  $0.9\%$ behind the best core, which is the only one clear of SCORE (Diag on MSE
  and CRPS, LorD on ES); the other two tie it. **Second on the variogram score**:
  ahead of Diag, tied with Chol and LorD.
* **ETTh2 and ETTm2: tied with both cores on every metric**, $2$ of $3$ in all
  eight cells. The pool sits inside $1.1\%$ on each metric, which is the case the
  tie rule exists for: the exact means would read 1st on CRPS and 3rd of 3 on
  MSE, ES and VS on ETTh2 ($0.33\%$ to $1.04\%$ behind the low-rank $r{=}1$
  core), and 1st on CRPS, 2nd on VS and 3rd on MSE and ES on ETTm2 ($0.22\%$ to
  $1.04\%$ behind the diagonal).

Two things follow for how these tables should be read. The rank is a fair
summary of *how often* SCORE is best and a poor one of *by how much*, so the
`gap` column of `ks_ranks_*` belongs next to it. And the variogram score is
again the column that behaves differently -- it is the one column on which no
core beats SCORE on any data set: first outright on ETTh1, ETTm1 and ERA5, tied
everywhere else, including EUPPBench and Depth, where the other three columns
put it behind. That is the same split section [2.4](#24-the-win-counts) found
between the marginal columns and the pairwise one.

Adding SB moves SCORE in seven cells: a full place on MSE,
CRPS and ES of EUPPBench, where SB is clear of it, and half a place
where the two tie -- MSE on ETTh2, ETTm2 and Depth, and VS on Depth. The row
goes from $2.14 / 1.86 / 1.93 / 1.57$ to $2.50 / 2.00 / 2.07 / 1.64$; everywhere
else SCORE is ahead of SB. Depth is the instructive one, because the
rank there is the ordinal summary at its least informative: SB's mean
is below SCORE's on MSE by $2.9\%$ and on VS by $1.4\%$, inside the seed spread,
while it is $18.4\%$ worse on CRPS and $20.8\%$ worse on ES -- it has collapsed
to a point forecast on every seed -- so under the exact means it took a
point-accuracy rank, and under the tie rule it does not even take that.

### 3.4 The LaTeX tables

Both single-row tables, written next to the others in `notebooks/tables/`. The
pool is in the caption rather than the table, since a single row has nowhere to
put it -- read the caption and the marginal table above together.

In [ ]:
_RANK_CAPTION_BASE = (
    r" Within each data set and metric the arms are ranked by their seed-mean"
    r" score, lower is better, arms whose mean $\pm$ standard deviation intervals"
    r" overlap at the precision of the per-task tables sharing their rank, and"
    r" the cell is that rank averaged over the"
    r" data sets; the arms are compared on the seeds they share. Every arm is fitted"
    r" with the kernel score, so the ranking is within one objective."
)

RANK_CAPTIONS = {
    "cores": (
        r"Mean rank of SCORE, the marginally standardized core, among the"
        r" kernel-score-fitted Gaussian baselines: Diag and LorD ($r = 1$) in"
        r" physical space, plus Chol on EUPPBench, the only data set where it"
        r" is tractable. Three arms on"
        r" ETTh1, ETTh2, ETTm1, ETTm2, ERA5 and NYU Depth v2, four on EUPPBench."
        r" The four ETT data sets are the multivariate tasks at $T = 96$."
        + _RANK_CAPTION_BASE),
    "sampler": (
        r"Mean rank of SCORE, the marginally standardized core, among the"
        r" kernel-score-fitted baselines with the sample-based arm included:"
        r" Diag and LorD ($r = 1$) in physical space, Chol on EUPPBench and"
        r" SB. Four arms on"
        r" ETTh1, ETTh2, ETTm1, ETTm2, ERA5 and NYU Depth v2, five on EUPPBench."
        r" The four ETT data sets are the multivariate tasks at $T = 96$."
        + _RANK_CAPTION_BASE),
}


def to_latex_rank_row(ranks, caption, label, row="SCORE", metrics=PAIR_METRICS):
    """The single row of `rank_row` as a `booktabs` table."""
    metrics = list(metrics)
    mean = ranks.groupby("metric")["rank"].mean().reindex(metrics)
    n = ranks.groupby("metric")["rank"].size().reindex(metrics).fillna(0).astype(int)
    cells = ["---" if n[c] == 0 else f"${mean[c]:.1f}$" for c in metrics]
    return "\n".join([
        r"\begin{table}[ht]", r"\centering",
        rf"\caption{{{caption}}}", rf"\label{{tab:{label}}}", r"\small",
        r"\setlength{\tabcolsep}{6pt}",
        r"\begin{tabular}{l" + "r" * len(metrics) + "}",
        r"\toprule",
        "Core & " + " & ".join(METRIC_LABEL[c] for c in metrics) + r" \\",
        r"\midrule",
        f"{row} & " + " & ".join(cells) + r" \\",
        r"\bottomrule", r"\end{tabular}", r"\end{table}",
    ])


TABLE_DIR.mkdir(parents=True, exist_ok=True)
for key, ranks, stem in (("cores", ks_ranks_cores, "ks_rank_ours"),
                         ("sampler", ks_ranks_sampler, "ks_rank_ours_sampler")):
    latex_row = to_latex_rank_row(ranks, RANK_CAPTIONS[key], f"ks-rank-{key}")
    print(latex_row, end="\n\n")
    out = TABLE_DIR / f"{stem}.tex"
    out.write_text(latex_row + "\n")
    print(f"written to {out.relative_to(ROOT)}\n")

### 3.5 How much, not only how often: improvement over the deterministic arm

A rank says SCORE was beaten, never by how much, and [3.3](#33-the-two-tables)
leans on that twice -- ETTh2, where the whole pool sits inside $1\%$, and
EUPPBench, where being last costs under $2\%$. This table is the complement: the
same arms and the same seed-matched means, expressed as the percentage by which
each one beats the **deterministic** arm, the point forecast that shares their
backbone and predicts no covariance at all:

$$
I_m(a, t) \;=\; 100 \;\frac{\bar s_m(\mathrm{det}, t) - \bar s_m(a, t)}
                           {\left|\bar s_m(\mathrm{det}, t)\right|},
\qquad
\overline{I}_m(a) \;=\; \frac{1}{|T(a,m)|} \sum_{t \in T(a,m)} I_m(a, t) ,
$$

positive meaning better, since all four metrics are lower-is-better.

**This is the one table here that crosses fitting objectives, and it does so on
purpose.** The reference is fitted with MSE and everything measured against it
with the kernel score, so the two halves of the table read differently. On `MSE`
the reference is the arm that optimized the column, which is what makes a gain
there small and worth having. On `CRPS`, `ES` and `VS` the reference is a Dirac,
and a proper score penalizes a point forecast by construction -- so the
$25$-$30\%$ in those columns is the price of having no predictive distribution
at all, not a statement about the covariance. What the paper argues about is the
spread *within* a column, not its level.

`Full covariance` is dropped: it exists on EUPPBench alone, so its row would
average one data set against the others' seven. Every remaining row has all
seven data sets on every metric; the counts print
underneath, and the bolding rule of [1.7](#17-the-latex-table) would keep a row
with fewer out of the bold.

In [ ]:
#: The arm every improvement is measured against, as a key of the pool below.
#: The deterministic arm is the same backbone with a mean-only head, so a row of
#: this table is what predicting a covariance at all buys; the section header
#: says what that costs in comparability.
IMPROVEMENT_REF = "deterministic"


def improvement_arm(method=None, method_prefix=None, rank=None, stage1="*",
                    objective=None):
    """`ks_arm` with the objective spelled out, since this pool has two."""
    def match(key, cfg):
        if method is not None and cfg.method != method:
            return False
        if method_prefix is not None and not cfg.method.startswith(method_prefix):
            return False
        if key[2] != rank:
            return False
        if stage1 != "*" and cfg.stage1 != stage1:
            return False
        return objective is None or cfg.objective() == objective
    return match


#: Section 3's pool without `phys_cholesky` -- EUPPBench only, so its mean would
#: be one data set against the others' seven -- plus the MSE-fitted reference.
IMPROVEMENT_POOL = [
    dict(key="deterministic", label="Det (reference)",
         match=improvement_arm(method="deterministic", objective="mse")),
    dict(key="phys_diag", label="Diag",
         match=improvement_arm(method="phys_diag", objective="kernel_score")),
    dict(key="phys_lora_r1", label=r"LorD $r{=}1$",
         match=improvement_arm(method="phys_lora", rank="1",
                               objective="kernel_score")),
    dict(key="generative", label="SB",
         match=improvement_arm(method="generative", objective="kernel_score")),
    dict(key="ours", label="SCORE",
         match=improvement_arm(method_prefix="whitened_", stage1="crps",
                               objective="kernel_score")),
]

IMPROVEMENT_ROWS = [m["key"] for m in IMPROVEMENT_POOL]
IMPROVEMENT_LABEL = {m["key"]: m["label"] for m in IMPROVEMENT_POOL}


def improvement_frame(scores, ref=IMPROVEMENT_REF):
    """One row per (task, metric, method): the % by which `method` beats `ref`.

    Built the same way as the ranks above -- seed means first, seeds matched
    across the pool of a cell -- so a number here and a rank there are the same
    aggregation of the same runs.
    """
    rows = []
    for (task, metric), grp in scores.groupby(["task", "metric"]):
        mean = grp.set_index("method")["mean"]
        if ref not in mean.index:
            print(f"no {ref} arm on {task}/{metric}: no reference, cell skipped")
            continue
        for method, value in mean.items():
            rows.append(dict(task=task, metric=metric, method=method,
                             improvement=100 * (mean[ref] - value) / abs(mean[ref])))
    return pd.DataFrame(rows)


def improvement_table(imp, how="mean", metrics=PAIR_METRICS):
    """Rows = methods, columns = metrics: the improvement pooled over data sets."""
    metrics = list(metrics)
    piv = imp.pivot_table(index="method", columns="metric", values="improvement",
                          aggfunc=how)
    cov = imp.pivot_table(index="method", columns="metric", values="improvement",
                          aggfunc="size")
    rows = [k for k in IMPROVEMENT_ROWS if k in piv.index]
    piv = piv.reindex(index=rows, columns=metrics)
    cov = cov.reindex(index=rows, columns=metrics).fillna(0).astype(int)
    for frame in (piv, cov):
        frame.index = [IMPROVEMENT_LABEL[i] for i in frame.index]
        frame.columns = [METRIC_LABEL[c].replace("$", "").replace("\\", "")
                         for c in metrics]
    return piv, cov


def improvement_marginal(imp, metrics=PAIR_METRICS):
    """Rows = (method, data set), columns = metrics -- the cells behind the mean."""
    metrics = list(metrics)
    order = [(m, t["key"]) for m in IMPROVEMENT_ROWS for t in TASKS]
    piv = (imp.pivot_table(index=["method", "task"], columns="metric",
                           values="improvement")
              .reindex(index=pd.MultiIndex.from_tuples(order), columns=metrics)
              .dropna(how="all"))
    piv.index = pd.MultiIndex.from_tuples(
        [(IMPROVEMENT_LABEL[m], next(t["title"] for t in TASKS if t["key"] == k))
         for m, k in piv.index])
    piv.columns = [METRIC_LABEL[c].replace("$", "").replace("\\", "") for c in metrics]
    return piv


imp_scores = pool_scores(runs, IMPROVEMENT_POOL)
improvements = improvement_frame(imp_scores)
imp_mean, imp_cov = improvement_table(improvements)

display(Markdown("**Mean improvement over Det, % of its "
                 "score** (positive = better)"))
display(imp_mean.round(3))
display(Markdown("**Data sets each mean is taken over**"))
display(imp_cov)
display(Markdown("**The median instead**, which is the same table with the "
                 "sampler's one bad data set taken out of its row"))
display(improvement_table(improvements, how="median")[0].round(3))
display(Markdown("**Per data set** -- the cells the means pool"))
display(improvement_marginal(improvements).round(3))

Three things the ranks could not show:

* **On the columns where the point forecast fails, the covariance cores are
  indistinguishable.** CRPS and ES put every Gaussian arm between $+26.3\%$ and
  $+28.4\%$, and inside either column the three sit within $1.4$ points of each
  other. Nearly all of that level is the Dirac penalty the reference pays for
  having no spread, and it is the same penalty for everyone.
* **VS is where the cores separate** -- $+7.0\%$ for the diagonal, $+8.3\%$ for
  the low-rank core and $+11.3\%$ for SCORE, about $1.6$ times the diagonal's
  gain on the one column that reads the pairwise structure. Depth is the one
  data set where none of them separates from the point forecast ($-0.3$ to
  $+1.5\%$, SCORE last), and it pulls the three means down by $1.1$ to $1.9$
  points. It is the
  split [2.4](#24-the-win-counts) and [3.3](#33-the-two-tables) found, seen as a
  margin instead of as a count.
* **MSE is flat by construction.** Every arm shares the backbone and the mean
  head with the reference, which was trained on this column, so every Gaussian
  cell of it lies in $[-3.3, +5.2]\%$ -- the top of that range is ETTm2, where
  every arm beats the MSE-fitted reference on its own column by $2.7$ to $5.2\%$;
  SCORE averages $+1.4\%$, held down by Depth's $-3.3\%$. The one cell outside it is
  SB on ETTm1, at $-7.0\%$.

The SB row is the one to read with the per-data-set matrix rather than
off its mean: $+18.8\%$ on CRPS and $+19.1\%$ on ES are $+25$ to $+30\%$ on ERA5
and EUPPBench, $+22$ to $+26\%$ on the two ETTm tasks, $+10$ to $+17\%$ on the two
ETTh tasks and $-0.3\%$ / $+0.2\%$ on Depth, where its ensemble has collapsed to
the point forecast it is measured against. The median row, $+24.0\%$ and
$+22.5\%$, is one data set's own value (ETTm1 on CRPS, ETTm2 on ES) and sits with
the upper five, while the mean is pulled down by Depth and ETTh2 -- the two
describe different things here, so neither summary stands alone.

In [ ]:
IMPROVEMENT_CAPTION = (
    r"Improvement over Det, the deterministic arm -- the same backbone with a"
    r" mean-only head -- in percent of its score and averaged over the data"
    r" sets; positive is better, since all four metrics are lower-is-better."
    r" The ranked arms are fitted with the kernel score and the reference with"
    r" MSE, so the columns read differently: on MSE the reference is the arm"
    r" that optimized the column, while on CRPS, ES and VS it is a point"
    r" forecast, which a proper score penalizes by construction. The arms of a"
    r" cell are compared on the seeds they share. Each mean is over all seven"
    r" data sets; ETTh1, ETTh2, ETTm1 and ETTm2 are the multivariate tasks at"
    r" $T = 96$."
    r" \textbf{Bold} is the best entry of a column among the rows averaged over"
    r" all of its data sets."
)


def to_latex_improvement_table(piv, cov, caption, label="improvement-det",
                               digits=3, ref=None):
    """The improvement table as `booktabs` LaTeX; the reference row prints ---."""
    ref = IMPROVEMENT_LABEL[IMPROVEMENT_REF] if ref is None else ref
    cols = list(piv.columns)
    full = {c: (cov[c] == cov[c].max()) & (piv.index != ref) for c in cols}
    best = {c: piv[c][full[c]].max() for c in cols}

    def cell(row, col):
        v = piv.loc[row, col]
        if row == ref or pd.isna(v):
            return "---"
        body = f"{v:+.{digits}f}"
        if abs(v - best[col]) < 10 ** -(digits + 3):
            body = rf"\mathbf{{{body}}}"
        return f"${body}$"

    lines = [
        r"\begin{table}[ht]", r"\centering",
        rf"\caption{{{caption}}}", rf"\label{{tab:{label}}}", r"\small",
        r"\setlength{\tabcolsep}{6pt}",
        r"\begin{tabular}{l" + "r" * len(cols) + "}",
        r"\toprule",
        "Core & " + " & ".join(cols) + r" \\",
        r"\midrule",
    ]
    lines += [f"{row} & " + " & ".join(cell(row, c) for c in cols) + r" \\"
              for row in piv.index]
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    return "\n".join(lines)


improvement_latex = to_latex_improvement_table(imp_mean, imp_cov, IMPROVEMENT_CAPTION)
print(improvement_latex)

TABLE_DIR.mkdir(parents=True, exist_ok=True)
improvement_path = TABLE_DIR / "improvement_vs_deterministic.tex"
improvement_path.write_text(improvement_latex + "\n")
print(f"\nwritten to {improvement_path.relative_to(ROOT)}")